# Registros afectados por variable

El propósito de este notebook es identificar la cantidad de registros afectados por cada variable editada, para completar la columna de registros afectados de la tabla ubicada en /docs/Registro_de_Transformaciones.md. Para lograrlo se compara el dataset crudo contra el dataset limpio, cruzando ambos por CODIGO, y se cuenta en cuántos registros cambió el valor de cada variable. En el caso de NIVEL, al ser una columna que se eliminó por completo, se reporta el total de filas.

In [1]:
from pathlib import Path

import pandas as pd

# Ubica data/ ya sea corriendo desde notebooks/ o desde la raiz del repo.
DIR_RAW = Path.cwd().parent / "data" / "raw"
if not DIR_RAW.exists():
    DIR_RAW = Path.cwd() / "data" / "raw"
RUTA_LIMPIO = DIR_RAW.parent / "processed" / "establecimientos_diversificado_limpio.csv"

crudo = pd.concat(
    [pd.read_csv(archivo, dtype=str) for archivo in sorted(DIR_RAW.glob("diversificado_*.csv"))],
    ignore_index=True,
)
limpio = pd.read_csv(RUTA_LIMPIO, dtype=str)

# Variables que se modificaron durante la limpieza, en el orden del pipeline.
variables_editadas = [
    "DIRECTOR", "DIRECCION", "AREA", "ESTABLECIMIENTO", "SUPERVISOR",
    "TELEFONO", "DISTRITO", "PLAN", "DEPARTAMENTAL", "NIVEL",
]

comparacion = crudo.merge(limpio, on="CODIGO", suffixes=("_crudo", "_limpio"))


def registros_afectados(variable):
    # NIVEL se elimino por completo, por lo que aplica a todas las filas.
    if variable not in limpio.columns:
        return len(limpio)
    antes = comparacion[f"{variable}_crudo"]
    despues = comparacion[f"{variable}_limpio"]
    # Un registro cambio si el valor difiere, tratando NaN == NaN como igual.
    iguales = (antes == despues) | (antes.isna() & despues.isna())
    return int((~iguales).sum())


resumen = pd.DataFrame({
    "variable": variables_editadas,
    "registros_afectados": [registros_afectados(variable) for variable in variables_editadas],
})
resumen

,variable,registros_afectados
0,DIRECTOR,61
1,DIRECCION,5
2,AREA,3
3,ESTABLECIMIENTO,5
4,SUPERVISOR,3
5,TELEFONO,201
6,DISTRITO,70
7,PLAN,8777
8,DEPARTAMENTAL,2025
9,NIVEL,11867
